<!-- DATA PROVIDER INSTRUCTIONS

1. Provide the name of your dataset, replacing the bracketed placeholder text.
2. Update the Registry of Open Data landing page URL, by replacing the bracketed placeholder text. The chammi-75 will correspond to the name of the YAML document in your pull request to the Registry of Open Data on Github, minus the .yaml file extension.
3. Remove these comment blocks when you have completed each section.

DATA PROVIDER INSTRUCTIONS -->

# Get to Know a Dataset: CHAMMI-75

This notebook serves as a guided tour of the CHAMMI-75 (https://registry.opendata.aws/chammi-75) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

<!-- DATA PROVIDER INSTRUCTIONS

The goal of this section is to orient users to the structure of your dataset. 

1. How are key prefixes and objects organized in your S3 bucket?
2. What kinds of filetypes are represented in your dataset?
3. Explain with text what users are expected to encounter, and then demonstrate with code the organizational framework you applied when creating your dataset.
4. The responses to each question section are meant to be expanded or replaced as dictated by your dataset

DATA PROVIDER INSTRUCTIONS -->

### Q: What is the overall structure of your dataset in S3?

The CHAMMI-75 dataset has the following components organized in the following way:
1. CHAMMI-75_train - This key prefix contains training data files in PNG format. Each file contains a single channel of the image separate. We have 75 different folders containing images from different sources of images for microscopy.
2. CHAMMI-75_test - This key prefix contains all the 5 different evaluation benchmarks for the CHAMMI-75 dataset. Each benchmark is stored in a separate folder.
3. License.txt - This file contains the license information for the dataset.
4. CHAMMI-75_guidance - This file contains all the different content guidance centroids required to succesfully train a model on the CHAMMI-75 dataset.
5. CHAMMI-75_train_metadata.csv - This file contains metadata information for each image in the dataset.
6. CHAMMI-75_small_metadata.csv - This file contains metadata information for each image in the smaller version of the dataset. 

### Q: What kinds of filetypes are represented in your dataset?
The CHAMMI-75 dataset contains the following filetypes:
1. PNG files - These files contain the microscopy images in the dataset.
2. ZIP files - This file contains the content guidance centroids required to succesfully train a model on the CHAMMI-75 dataset.
3. CSV files - These files contain metadata information for each image in the dataset.
4. TXT files - This file contains the license information for the dataset.

Full documentation for this dataset can be found in a research publication at: [ARXIV LINK TO BE ADDED AFTER RELEASE]

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of our S3 bucket, we have all the different objects stated in Q1. The training and testing data are organized in separate key prefixes as stated in Q1. Each training image is stored in a separate PNG file, and each testing benchmark is stored in a separate folder containing the respective images.
 
Full documentation for this dataset can be found at: [ARXIV LINK TO BE ADDED AFTER RELEASE]

In [ ]:
# CODING GUIDELINES FOR DATA PROVIDER
#
# General notebook coding guidelines:
# 1. Assume that your reader understands the basics of Jupyter Notebooks, Python, and their Python environment.
#    The focus of this tutorial is on your dataset.
# 2. For library requirements, list the required libraries in a comment block in "requirements.txt" format
#    (https://pip.pypa.io/en/stable/reference/requirements-file-format/)
# 3. Demonstrate importing libraries with the assumption that the user has correctly installed the required
#    libraries.
# 4. List and load all library dependencies once, at this point of the notebook, unless a complicated dependency
#    set makes it unweildy.
# 5. Remember, the goal of this tutorial is a 101-level introduction to your dataset using common tools and libraries.
#    Examples using specialized environments and deep-diving methods are better suited to follow-up tutorials.
#
# CODING GUIDELINES FOR DATA PROVIDER

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# matplotlib >= 3.10.3
# skimage

# Import the libraries required for this notebook
# Built-ins

# Installed libraries
import boto3
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config
import pandas as pd
import s3fs
from PIL import Image
from io import BytesIO

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see there is only one top-level prefix in our bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "chammi-data"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

# List all items in the bucket
response = s3.list_objects_v2(Bucket=bucket, Delimiter="/")

# Print folders
if "CommonPrefixes" in response:
    for item in response["CommonPrefixes"]:
        print(item["Prefix"])

# Print files
if "Contents" in response:
    for item in response["Contents"]:
        print(item["Key"])

Load the small dataset metadata file, and showcase some basic information about the data.

In [ ]:
# Create an S3 filesystem object
s3 = s3fs.S3FileSystem(anon=True)  # anon=False uses your AWS credentials

# Read CSV directly from S3
df = pd.read_csv("s3://chammi-data/CHAMMI-75/CHAMMI-75_small_metadata.csv")
df.columns

Show the names of all the different dataset folders present in the dataset. The training and evaluation prefixes are similar in structure, and so we can look into the training portion to get a sense of the deeper structure of the dataset where the data objects reside.

In [ ]:
# Create a boto3 S3 client

# List all items inside the CHAMMI-75_train folder
response = s3.list_objects_v2(
    Bucket="chammi-data", Prefix="CHAMMI-75_train/", Delimiter="/"
)

# Print folders (prefixes) inside CHAMMI-75_train/
if "CommonPrefixes" in response:
    print("Dataset Names inside CHAMMI-75 Pre-training set:")
    datasets = [item["Prefix"].split("/")[-2] for item in response["CommonPrefixes"]]

    for i in range(0, len(datasets), 10):
        row = datasets[i : i + 10]
        print("  " + "  ".join(f"{d:10}" for d in row))

<!-- DATA PROVIDER INSTRUCTIONS
This section is meant to orient users of your dataset to the formats present in your dataset, particularly if your dataset includes formats that may be unfamiliar to a general data scientist audience. This section should include:

1. Explanation of data format(s) (very common formats can be very briefly described, while less common
   or domain specific formats should include more explanation as well as links to official documentation)
2. Explanation of why the data format was chosen for your dataset
3. Recommendations around software and tooling to work with this data format
4. Explanation of any dataset-specific aspects to your usage of the format
5. Description of AWS services that may be useful to users working with your data
DATA PROVIDER INSTRUCTIONS -->

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?


Our dataset has PNG files and contains in total 2.8 million multi-channel images (8 million single channel images) preprocessed for investigation of foundations models in cellular morphology.

The documentation can be found at https://github.com/CaicedoLab/CHAMMI-75


<!-- DATA PROVIDER INSTRUCTIONS
The goal of this section is to demonstrate loading a portion of data from your dataset, and reveal something about its structure.
1. Load an object from S3
2. Show the structure of data in the object
DATA PROVIDER INSTRUCTIONS -->

### Q: Can you show us an example of downloading and loading data from your dataset?

We will show a load up example once the dataset is uploaded. We will be loading a sample PNG image from HPA0018.

In [ ]:
# Image paths
image_keys = [
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch0.png",
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch1.png",
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch2.png",
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch3.png",
]

# Download each image
for key in image_keys:
    # Extract filename from the key
    filename = key.split("/")[-1]

    # Download the file
    s3.download_file("chammi-data", key, filename)
    print(f"Downloaded: {filename}")

<!-- DATA PROVIDER INSTRUCTIONS
The goal here is to visualize some aspect of your dataset in order to help users understand it. In addition to helping users of your dataset understand the dataset, an additional goal is to impress!

Please demonstrate any data preprocessing or reshaping required for your visualization(s).

https://www.reddit.com/r/dataisbeautiful/ for inspiration.
DATA PROVIDER INSTRUCTIONS -->

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Visualizating some images of the dataset.

In [ ]:
# Image paths
image_keys = [
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch0.png",
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch1.png",
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch2.png",
    "CHAMMI-75_train/hpa0018/00070df0-bbc3-11e8-b2bc-ac1f6b6435d0-ch3.png",
]

# Load images from S3
images = []
for key in image_keys:
    obj = s3.get_object(Bucket="chammi-data", Key=key)
    img = Image.open(BytesIO(obj["Body"].read()))
    images.append(img)

# Visualize all 4 channels together
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (img, ax) in enumerate(zip(images, axes)):
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Channel {idx}")
    ax.axis("off")

plt.tight_layout()
plt.show()

<!-- DATA PROVIDER INSTRUCTIONS
This section is less prescriptive / freeform than previous sections. The goal here is to show an opinionated example of answering a question using your data. The scale of your dataset may preclude a full example, and so feel free to limit the scope of this example (e.g. work on a subset of data). Users should be able to replicate your example in this notebook, and get a sense of how they would scale up.

A "toy" example is better than no example.

Ideally, your example would:
1. Transmit some of your domain & dataset experience to the reader, drawing on your own work as much as possible
2. Provide a jumping off point for users to extend your work, and do novel work of their own.

DATA PROVIDER INSTRUCTIONS -->

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

This dataset can be used for pre-training foundation models for cellular morphology. It can also be used to download evaluation benchmarks for evaluating models trained on cellular morphology tasks. The tutorial for that is the next notebook in this series.

<!-- DATA PROVIDER INSTRUCTIONS
This section is, like the previous one, intended to be freeform / non-prescriptive. The goal here is to provide a challenge to the community to do something novel with your dataset. That can either be novel in terms of the task, or novel in terms of methodological or computational approach.

Another way to consider this section, is as a wishlist. If you were less constrained by time, cost, skill, etc., what would you like to see achieved using these data? 

The challenge should, however, be somewhat realistic. A challenge that assumes e.g. original data collection, is likely to go unanswered.
DATA PROVIDER INSTRUCTIONS -->

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

This large dataset can be used for generative modeling of cellular morphology using diffusion models or other generative modeling approaches.